# Tutorial: Purview Sensitivity Labels with SharePoint Knowledge Source

This notebook creates a SharePoint-indexed agentic retrieval pipeline on Azure AI Search that surfaces Microsoft Purview sensitivity labels on every retrieved document.

**Flow:**
1. Create an `indexedSharePoint` knowledge source with `sensitivityLabels` ingestion — triggers an internal async pipeline that crawls SharePoint and stores each document's Purview label.
2. Poll until the knowledge source has completed its first sync.
3. Create a knowledge base backed by the SharePoint knowledge source.
4. Retrieve with a permission-filtered access token and inspect `PurviewSensitivityLabelInfo` on each reference.
5. Clean up.

**SDK:** `azure-search-documents==12.1.0b1` (REST API `2026-05-01-preview`)

For prerequisites and setup instructions, save `sample.env` as `.env` and fill in the values, then create a virtual environment from `requirements.txt`.

## Load connections

Before you run this cell, save `sample.env` as `.env` and fill in the values. You should also create a virtual environment with `purview-sharepoint-labels-example/requirements.txt` as the dependencies.

In [ ]:
import os
import time

from azure.identity import ClientSecretCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint            = os.environ["AZURE_SEARCH_ENDPOINT"]
client_id           = os.environ["AZURE_CLIENT_ID"]
client_secret       = os.environ["AZURE_CLIENT_SECRET"]
tenant_id           = os.environ["AZURE_TENANT_ID"]
sp_ks_name          = os.getenv("AZURE_SEARCH_KS_NAME", "purview-sharepoint-ks")
kb_name             = os.getenv("AZURE_SEARCH_KB_NAME", "purview-sharepoint-kb")
sharepoint_site_url = os.environ["SHAREPOINT_SITE_URL"]
openai_endpoint     = os.environ["AZURE_OPENAI_ENDPOINT"]
openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")
openai_embedding_model      = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
chat_endpoint       = os.environ["AZURE_OPENAI_CHAT_ENDPOINT"]
chat_api_key        = os.environ["AZURE_OPENAI_CHAT_API_KEY"]
chat_deployment     = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")
chat_model          = os.getenv("AZURE_OPENAI_CHAT_MODEL", "gpt-4o-mini")

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret,
)

# SharePoint Online connection string for the indexed knowledge source pipeline
sp_connection_string = (
    f"SharePointOnlineEndpoint={sharepoint_site_url};"
    f"ApplicationId={client_id};"
    f"ApplicationSecret={client_secret};"
    f"TenantId={tenant_id}"
)

index_client        = SearchIndexClient(endpoint=endpoint, credential=credential)
kb_retrieval_client = KnowledgeBaseRetrievalClient(endpoint=endpoint, credential=credential)

print(f"Endpoint:         {endpoint}")
print(f"Knowledge source: {sp_ks_name}")
print(f"Knowledge base:   {kb_name}")

## Create an indexedSharePoint knowledge source

This knowledge source triggers an internal async indexing pipeline that crawls the specified SharePoint document library and stores Purview sensitivity labels alongside the document vectors.

Key parameters:
- `ingestion_permission_options=[SENSITIVITY_LABELS]` — reads and stores the Purview label on every ingested document.
- `content_extraction_mode="minimal"` — extracts metadata only; use `"default"` to also index full text.
- `embedding_model` — Azure OpenAI vectorizer used to embed documents at ingestion time.
- `ingestion_schedule` — recurring sync cadence as an ISO 8601 duration.

The response includes `created_resources` — the internal index and indexer provisioned by the service.

In [ ]:
from azure.search.documents.indexes.models import (
    IndexedSharePointKnowledgeSource,
    IndexedSharePointKnowledgeSourceParameters,
    KnowledgeSourceIngestionPermissionOption,
    IndexingSchedule,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceIngestionParameters,
    KnowledgeSourceAzureOpenAIVectorizer,
)

ks = IndexedSharePointKnowledgeSource(
    name=sp_ks_name,
    indexed_share_point_parameters=IndexedSharePointKnowledgeSourceParameters(
        connection_string=sp_connection_string,
        container_name="defaultSiteLibrary",
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            ingestion_permission_options=[
                KnowledgeSourceIngestionPermissionOption.SENSITIVITY_LABELS
            ],
            content_extraction_mode="minimal",
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                resource_uri=openai_endpoint,
                deployment_id=openai_embedding_deployment,
                model_name=openai_embedding_model,
            ),
            ingestion_schedule=IndexingSchedule(interval="P1D"),
        ),
    ),
)

result = index_client.create_or_update_knowledge_source(knowledge_source=ks)
print(f"Knowledge source '{result.name}' created.")
if getattr(result, "created_resources", None):
    for r in result.created_resources:
        print(f"  Created resource: {r}")

## Poll knowledge source sync status

The `indexedSharePoint` pipeline is asynchronous. The cell below polls every 30 seconds until the knowledge source reports a terminal sync status (`success` or `error`). Initial sync for a large library may take several minutes.

If `sync_status` prints as `None`, inspect the full `ks_status` object to identify the correct status field for your SDK build.

In [ ]:
POLL_INTERVAL_SECS = 30
POLL_TIMEOUT_SECS  = 1800  # 30 minutes

start      = time.time()
sync_status = None
while True:
    ks_status   = index_client.get_knowledge_source(sp_ks_name)
    sync_status = (
        getattr(ks_status, "last_sync_status", None)
        or getattr(ks_status, "status", None)
    )
    elapsed = int(time.time() - start)
    print(f"[{elapsed:4d}s] sync_status={sync_status}")
    if sync_status in ("success", "error", "transientFailure"):
        break
    if time.time() - start > POLL_TIMEOUT_SECS:
        print("Timed out waiting for sync. Proceeding anyway.")
        break
    time.sleep(POLL_INTERVAL_SECS)

print(f"Final sync status: {sync_status}")

## Create a knowledge base

The knowledge base wraps the SharePoint knowledge source and provides the Azure OpenAI chat model used for answer synthesis. `ANSWER_SYNTHESIS` mode returns a grounded answer together with Purview-labelled document references.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalMinimalReasoningEffort,
)

kb = KnowledgeBase(
    name=kb_name,
    knowledge_sources=[KnowledgeSourceReference(name=sp_ks_name)],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    models=[
        KnowledgeBaseAzureOpenAIModel(
            resource_uri=chat_endpoint,
            api_key=chat_api_key,
            deployment_id=chat_deployment,
            model_name=chat_model,
        )
    ],
)

index_client.create_or_update_knowledge_base(knowledge_base=kb)
print(f"Knowledge base '{kb_name}' created or updated successfully")

## Retrieve with permission filtering

Retrieval against a Purview-enabled knowledge source requires a raw Azure AI Search access token in `x-ms-query-source-authorization`. The service uses this token to enforce document-level permissions and to attach the Purview sensitivity label to each returned reference.

> **Important**: pass the raw JWT — do **not** include a `Bearer` prefix.

The response contains:
- `response_sensitivity_label_info` — the highest-sensitivity label across all referenced documents (`PurviewSensitivityLabelInfo`).
- `references[n].search_sensitivity_label_info` — per-document `PurviewSensitivityLabelInfo` with `label_id` and `label_name`.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalMinimalReasoningEffort,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    IndexedSharePointKnowledgeSourceParams,
)

# Acquire a raw Search access token — no "Bearer" prefix.
raw_token = credential.get_token("https://search.azure.com/.default").token

query = "Find SharePoint documents and include their file names and sensitivity label information."

request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=query)],
        )
    ],
    knowledge_source_params=[
        IndexedSharePointKnowledgeSourceParams(
            knowledge_source_name=sp_ks_name,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
            max_output_documents=50,
        )
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    include_activity=True,
)

response = kb_retrieval_client.retrieve(
    knowledge_base_name=kb_name,
    body=request,
    x_ms_query_source_authorization=raw_token,
)

# Response-level sensitivity label (highest sensitivity across all referenced documents)
resp_label = getattr(response, "response_sensitivity_label_info", None)
if resp_label:
    print(f"Response sensitivity label: {resp_label.label_name} ({resp_label.label_id})")
else:
    print("Response sensitivity label: (none)")

print(f"\nAnswer:\n{response.response}")

# Per-document sensitivity labels
references = getattr(response, "references", []) or []
print(f"\nReferences ({len(references)}):")
for ref in references:
    sl         = getattr(ref, "search_sensitivity_label_info", None)
    label_str  = f"{sl.label_name} ({sl.label_id})" if sl else "(no label)"
    source_data = getattr(ref, "source_data", {}) or {}
    doc_name   = (
        source_data.get("metadata_storage_name")
        or source_data.get("title")
        or "unknown"
    )
    print(f"  {doc_name:<40s}  label={label_str}")

## Clean up objects

Delete the knowledge base and knowledge source. Deleting the knowledge source also removes the internal index and indexer provisioned by the `indexedSharePoint` pipeline.

### Delete the knowledge base

In [ ]:
index_client.delete_knowledge_base(kb_name)
print(f"Knowledge base '{kb_name}' deleted successfully")

### Delete the knowledge source

In [ ]:
index_client.delete_knowledge_source(knowledge_source=sp_ks_name)
print(f"Knowledge source '{sp_ks_name}' deleted successfully")